In [ ]:
import json 
import pandas as pd 
import requests
import time
import random

from typing import Optional
from pathlib import Path
from bs4 import BeautifulSoup
from collections import defaultdict

df = pd.read_csv("../../datasets_clean.csv")

# df_single_gse = df[df["response"].str.split(",").str.len() == 1]
# df_single_gse = df_single_gse[df_single_gse["response"].str.startswith("GSE")]
# df_single_gse = df_single_gse.set_index("pmcid")
# print(df_single_gse.shape)

# pmcid_to_gse = df_single_gse["response"].to_dict()
# print(len(pmcid_to_gse))


# Get all GSE IDs from all responses (including comma-separated ones)
flattened_gse_ids = list(set([part.strip() for val in df['response'] for part in str(val).split(',') if part.strip().startswith('GSE')]))
print(f"Total unique GSE IDs found: {len(flattened_gse_ids)}")

# Create mapping from PMCID to all GSE IDs in that response
pmcid_to_gse_list = {}
for _, row in df.iterrows():
    pmcid = row['pmcid']
    response = str(row['response'])
    gse_ids = [part.strip() for part in response.split(',') if part.strip().startswith('GSE')]
    if gse_ids:
        pmcid_to_gse_list[pmcid] = gse_ids

print(f"PMCIDs with GSE responses: {len(pmcid_to_gse_list)}")

# Create GSE to PMCID mapping (each GSE can map to multiple PMCIDs)
gse_to_pmcids = defaultdict(list)
for pmcid, gse_ids in pmcid_to_gse_list.items():
    for gse_id in gse_ids:
        gse_to_pmcids[gse_id].append(pmcid)

# Preserve existing canonical/curated entries while adding newly discovered GSEs.
mapping_path = Path("data/gse_to_pmcids.json")
if mapping_path.exists():
    with mapping_path.open(encoding="utf-8") as f:
        existing_gse_to_pmcids = json.load(f)
    gse_to_pmcids.update(existing_gse_to_pmcids)

# Sort by number of associated PMCIDs (descending).
gse_to_pmcids = dict(sorted(gse_to_pmcids.items(), key=lambda x: len(x[1]), reverse=True))

with mapping_path.open("w", encoding="utf-8") as f:
    json.dump(gse_to_pmcids, f, indent=2)


Total unique GSE IDs found: 397
PMCIDs with GSE responses: 358


In [ ]:

# flattened_gse_ids = list(set([part.strip() for val in df['response'] for part in str(val).split(',') if part.strip().startswith('GSE')]))

def extract_experiment_type(html: str) -> Optional[str]:
    """
    Return the value for 'Experiment type' from a GEO accession HTML page,
    or None if not found.
    """
    soup = BeautifulSoup(html, "html.parser")

    # Find the <td> whose text is (or starts with) "Experiment type"
    label_td = soup.find(
        lambda tag: tag.name == "td"
        and tag.get_text(strip=True).lower().startswith("experiment type")
    )
    if not label_td:
        return None

    # Get the row (<tr>) containing that label and take the next <td>
    row = label_td.find_parent("tr")

    if not row:
        return None

    tds = row.find_all("td")
    if len(tds) < 2:
        return None

    # Extract text from the second cell; join fragments and strip whitespace
    value = tds[1].get_text(separator=", ", strip=True)
    return value


def check_gse_design(html: str) -> str:
    if "This SuperSeries is composed of the following SubSeries:" in html:
        return "is superseries"

    if "This SubSeries is part of SuperSeries:" in html:
        return "is subseries"

    return "is standalone"


def check_ncbi_generated_data(html: str) -> str:
    return "NCBI-generated data" in html

    
# with open("gse_metadata.tsv", "w", encoding="utf-8") as f:
#     f.write("gse_id\tdesign\tncbi_generated_data\texperiment_type\n")


df_processed = pd.read_csv("data/gse_metadata.tsv", sep="\t")
gse_ids_processed = set(df_processed['gse_id'])

for gse_id in flattened_gse_ids:
    if gse_id in gse_ids_processed:
        continue

    response = requests.get(f"https://www.ncbi.nlm.nih.gov/geo/query/acc.cgi?acc={gse_id}", timeout=20)
    time.sleep(random.uniform(0, 1))  
    response_download_page = requests.get(f"https://www.ncbi.nlm.nih.gov/geo/download/?acc={gse_id}", timeout=20)

    exp_type = extract_experiment_type(response.text)
    design = check_gse_design(response.text)
    ncbi_generated_data = check_ncbi_generated_data(response_download_page.text)
    with open("data/gse_metadata.tsv", "a", encoding="utf-8") as f:
        f.write(f"{gse_id}\t{design}\t{ncbi_generated_data}\t{exp_type}\n")
    time.sleep(random.uniform(1.0, 2.5))
    

In [6]:
df_metadata = pd.read_csv("data/gse_metadata.tsv", sep="\t")
df_metadata

,gse_id,design,ncbi_generated_data,experiment_type
0,GSE165334,is standalone,False,Expression profiling by high throughput sequen...
1,GSE201868,is standalone,True,Expression profiling by high throughput sequen...
2,GSE143222,is standalone,False,Expression profiling by array
3,GSE19776,is superseries,False,Expression profiling by array
4,GSE136961,is standalone,True,Expression profiling by high throughput sequen...
...,...,...,...,...
392,GSE90866,is standalone,False,Expression profiling by high throughput sequen...
393,GSE31684,is standalone,False,Expression profiling by array
394,GSE206199,is standalone,True,Expression profiling by high throughput sequen...
395,GSE140587,is superseries,False,Non-coding RNA profiling by array


# filter only highthroughput experiments 

In [8]:
df_high_throughput = df_metadata[
    df_metadata['experiment_type'].str.contains("Expression profiling by high throughput sequencing|Non-coding RNA profiling by high throughput sequencing")
].reset_index(drop=True)

# print(df_high_throughput["ncbi_generated_data"].sum())
# df_high_throughput.sort_values(by="ncbi_generated_data", ascending=False)

df_high_throughput.to_csv("data/gse_metadata-filtered.tsv", sep="\t", index=False)